<a href="https://colab.research.google.com/github/Sri-Hasini/Deep_Learning_Based_Medical_Image_Fusion/blob/main/Copy_of_Inhouse_Internship_2_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **DEEP LEARNING BASED MEDICAL IMAGE FUSION**

**SECTION-1  -> INSTALL LIBRARIES**

In [ ]:
!pip install torch torchvision
!pip install opencv-python
!pip install grad-cam

**SECTION-2 -> IMPORT LIBRARIES**

In [ ]:
import torch
import torch.nn as nn
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ======================================================================
# CELL 16: DOWNLOAD & EXTRACT REGISTERED HARVARD CT-MRI DATASET
# ======================================================================
import os
import shutil
import zipfile
import urllib.request
import ssl

# 1. Define folder paths to match your existing notebook
ct_folder = "/content/MedicalFusion/CT"
mri_folder = "/content/MedicalFusion/MRI"

# Clear existing folders if they exist to avoid contamination
if os.path.exists("/content/MedicalFusion"):
    shutil.rmtree("/content/MedicalFusion")

os.makedirs(ct_folder, exist_ok=True)
os.makedirs(mri_folder, exist_ok=True)

# 2. Download the curated Harvard Medical Image Fusion Dataset
url = "https://github.com/xianming-gu/Havard-Medical-Image-Fusion-Datasets/archive/refs/heads/master.zip"
zip_path = "/content/dataset.zip"

print("Downloading Harvard CT-MRI dataset from GitHub...")
ssl_context = ssl._create_unverified_context()
req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(req, context=ssl_context) as response:
    with open(zip_path, "wb") as f:
        f.write(response.read())
print("Download complete!")

# 3. Scan the zip file and extract ONLY the matching pairs
print("Extracting co-registered CT-MRI slices...")
with zipfile.ZipFile(zip_path, 'r') as z:
    namelist = z.namelist()

    # Exact prefixes in the zip archive to avoid duplicate subfolder files
    ct_prefix = "Havard-Medical-Image-Fusion-Datasets-main/CT-MRI/CT/"
    mri_prefix = "Havard-Medical-Image-Fusion-Datasets-main/CT-MRI/MRI/"

    # Identify files in the main CT and MRI folders of the zip
    ct_files = [name for name in namelist if name.startswith(ct_prefix) and name.endswith(".png")]
    mri_files = [name for name in namelist if name.startswith(mri_prefix) and name.endswith(".png")]

    # Extract only filenames (e.g., "16003.png")
    ct_filenames = {os.path.basename(f) for f in ct_files if os.path.basename(f)}
    mri_filenames = {os.path.basename(f) for f in mri_files if os.path.basename(f)}

    # Find files present in both modalities to guarantee perfect slice matching
    matching_filenames = sorted(list(ct_filenames.intersection(mri_filenames)))
    print(f"Found {len(matching_filenames)} perfectly aligned CT-MRI pairs.")

    # Extract matching images to the target directories
    for name in namelist:
        basename = os.path.basename(name)
        if basename in matching_filenames:
            if name.startswith(ct_prefix):
                with z.open(name) as source, open(os.path.join(ct_folder, basename), "wb") as target:
                    shutil.copyfileobj(source, target)
            elif name.startswith(mri_prefix):
                with z.open(name) as source, open(os.path.join(mri_folder, basename), "wb") as target:
                    shutil.copyfileobj(source, target)

# 4. Clean up downloaded ZIP
if os.path.exists(zip_path):
    os.remove(zip_path)

print("Extraction and setup complete!")
print("CT images count:", len(os.listdir(ct_folder)))
print("MRI images count:", len(os.listdir(mri_folder)))

**SECTION-3 -> LOAD DATASET**

In [ ]:
import os

ct_folder = "/content/MedicalFusion/CT"
mri_folder = "/content/MedicalFusion/MRI"

ct_images = sorted(os.listdir(ct_folder))
mri_images = sorted(os.listdir(mri_folder))

print("CT:", len(ct_images))
print("MRI:", len(mri_images))

**SECTION-4 ->PREPROCESSING**

In [ ]:
# Step 1: Define preprocessing function
import cv2
import numpy as np

def preprocess_image(path):

    # Read image in grayscale
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

    # Resize to fixed size
    img = cv2.resize(img, (256, 256))

    # CLAHE Enhancement
    clahe = cv2.createCLAHE(
        clipLimit=3.0,
        tileGridSize=(8, 8)
    )

    img = clahe.apply(img)

    # Normalize pixel values
    img = img.astype(np.float32) / 255.0

    return img

In [ ]:
#Step 2: Preprocess all images
ct_data = []
mri_data = []

for img_name in ct_images:
    path = os.path.join(ct_folder, img_name)
    ct_data.append(preprocess_image(path))

for img_name in mri_images:
    path = os.path.join(mri_folder, img_name)
    mri_data.append(preprocess_image(path))

ct_data = np.array(ct_data)
mri_data = np.array(mri_data)

print("CT Shape:", ct_data.shape)
print("MRI Shape:", mri_data.shape)

**SECTION-5 ->CNN ENCODER**

In [ ]:
# Step 1: Add Channel Dimension
import numpy as np

ct_data = np.expand_dims(ct_data, axis=1)
mri_data = np.expand_dims(mri_data, axis=1)

print("CT Shape :", ct_data.shape)
print("MRI Shape:", mri_data.shape)

In [ ]:
# Step 2: Import PyTorch
import torch
import torch.nn as nn

In [ ]:
import torch
import torch.nn as nn


# =====================================
# Multi-Scale Feature Extraction Block
# =====================================

class MultiScaleBlock(nn.Module):

    def __init__(self, in_channels, out_channels):

        super().__init__()

        self.branch3 = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.1, inplace=True)
        )

        self.branch5 = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=5,
                padding=2
            ),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.1, inplace=True)
        )

        self.branch7 = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=7,
                padding=3
            ),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.1, inplace=True)
        )

    def forward(self, x):

        f1 = self.branch3(x)
        f2 = self.branch5(x)
        f3 = self.branch7(x)

        return torch.cat([f1, f2, f3], dim=1)


# =====================================
# Residual Block
# =====================================

class ResidualBlock(nn.Module):

    def __init__(self, channels):

        super().__init__()

        self.conv1 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=1
        )

        self.bn1 = nn.BatchNorm2d(channels)

        self.conv2 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=1
        )

        self.bn2 = nn.BatchNorm2d(channels)

        self.act = nn.LeakyReLU(
            0.1,
            inplace=True
        )

    def forward(self, x):

        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.act(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + identity

        out = self.act(out)

        return out


# =====================================
# Dual Branch Encoder
# =====================================

class DualBranchEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        # Multi-scale extraction
        self.ms_block1 = MultiScaleBlock(
            in_channels=1,
            out_channels=16
        )

        self.pool1 = nn.MaxPool2d(2)

        # Feature compression
        self.conv1 = nn.Sequential(
            nn.Conv2d(
                48,
                128,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(
                0.1,
                inplace=True
            )
        )

        # Residual refinement
        self.resblock = ResidualBlock(128)

        self.pool2 = nn.MaxPool2d(2)

    def forward(self, x):

        # Input: [B,1,256,256]

        x = self.ms_block1(x)

        # [B,48,256,256]

        x = self.pool1(x)

        # [B,48,128,128]

        x = self.conv1(x)

        # [B,128,128,128]

        x = self.resblock(x)

        encoder_skip = x

        x = self.pool2(x)

        # [B,128,64,64]

        return x, encoder_skip

In [ ]:
# =====================================
# Create CT and MRI Encoders
# =====================================

ct_encoder = DualBranchEncoder()
mri_encoder = DualBranchEncoder()


# =====================================
# Convert Dataset to Tensors
# =====================================

import torch

ct_tensor = torch.tensor(
    ct_data,
    dtype=torch.float32
)

mri_tensor = torch.tensor(
    mri_data,
    dtype=torch.float32
)

print("CT Tensor Shape :", ct_tensor.shape)
print("MRI Tensor Shape:", mri_tensor.shape)


# =====================================
# DataLoader
# =====================================

from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(
    ct_tensor,
    mri_tensor
)

trainloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=True
)

print("Number of batches:", len(trainloader))

**SECTION-6 -> CROSS MODAL TRANSFORMER**

In [ ]:
import torch
import torch.nn as nn


# =====================================
# Feature Tokenizer
# =====================================

class FeatureTokenizer(nn.Module):

    def __init__(self,
                 in_channels=128,
                 embed_dim=128):

        super().__init__()

        self.proj = nn.Sequential(
            nn.Conv2d(
                in_channels,
                embed_dim,
                kernel_size=1
            ),

            nn.MaxPool2d(2)
        )

    def forward(self, x):

        x = self.proj(x)

        B, E, H, W = x.shape

        x = x.flatten(2).transpose(1, 2)

        return x, H, W


# =====================================
# Cross Attention
# =====================================

class CrossAttention(nn.Module):

    def __init__(self,
                 dim,
                 heads=4):

        super().__init__()

        self.heads = heads

        self.scale = (
            dim // heads
        ) ** -0.5

        self.to_q = nn.Linear(dim, dim)

        self.to_k = nn.Linear(dim, dim)

        self.to_v = nn.Linear(dim, dim)

        self.proj = nn.Linear(dim, dim)

    def forward(self,
                x,
                context):

        B, N, C = x.shape

        Q = self.to_q(x)
        K = self.to_k(context)
        V = self.to_v(context)

        Q = Q.view(
            B,
            N,
            self.heads,
            C // self.heads
        ).transpose(1, 2)

        K = K.view(
            B,
            N,
            self.heads,
            C // self.heads
        ).transpose(1, 2)

        V = V.view(
            B,
            N,
            self.heads,
            C // self.heads
        ).transpose(1, 2)

        attn = (
            Q @ K.transpose(-2, -1)
        ) * self.scale

        attn = attn.softmax(dim=-1)

        out = attn @ V

        out = out.transpose(
            1,
            2
        ).contiguous().view(
            B,
            N,
            C
        )

        return self.proj(out)


# =====================================
# Transformer Block
# =====================================

class TransformerBlock(nn.Module):

    def __init__(self,
                 dim,
                 heads=4,
                 mlp_dim=256):

        super().__init__()

        self.self_attn = nn.MultiheadAttention(
            dim,
            heads,
            batch_first=True
        )

        self.cross_attn = CrossAttention(
            dim,
            heads
        )

        self.norm1 = nn.LayerNorm(dim)

        self.norm2 = nn.LayerNorm(dim)

        self.norm3 = nn.LayerNorm(dim)

        self.mlp = nn.Sequential(

            nn.Linear(
                dim,
                mlp_dim
            ),

            nn.GELU(),

            nn.Dropout(0.1),

            nn.Linear(
                mlp_dim,
                dim
            ),

            nn.Dropout(0.1)
        )

    def forward(self,
                x,
                context):

        residual = x

        x = self.norm1(x)

        x, _ = self.self_attn(
            x,
            x,
            x
        )

        x = x + residual

        residual = x

        x = self.norm2(x)

        x = self.cross_attn(
            x,
            context
        )

        x = x + residual

        residual = x

        x = self.norm3(x)

        x = self.mlp(x)

        x = x + residual

        return x


# =====================================
# Cross Modal Transformer
# =====================================

class CrossModalTransformer(nn.Module):

    def __init__(self,
                 dim=128,
                 depth=2,
                 heads=4):

        super().__init__()

        self.ct_tokenizer = FeatureTokenizer(
            128,
            dim
        )

        self.mri_tokenizer = FeatureTokenizer(
            128,
            dim
        )

        self.layers = nn.ModuleList([

            TransformerBlock(
                dim,
                heads
            )

            for _ in range(depth)

        ])

    def forward(self,
                ct_feat,
                mri_feat):

        ct_tokens, H, W = self.ct_tokenizer(
            ct_feat
        )

        mri_tokens, _, _ = self.mri_tokenizer(
            mri_feat
        )

        for layer in self.layers:

            ct_tokens = layer(
                ct_tokens,
                mri_tokens
            )

            mri_tokens = layer(
                mri_tokens,
                ct_tokens
            )

        return (
            ct_tokens,
            mri_tokens,
            H,
            W
        )


# =====================================
# Test
# =====================================

if __name__ == "__main__":

    ct_feat = torch.randn(
        1,
        128,
        64,
        64
    )

    mri_feat = torch.randn(
        1,
        128,
        64,
        64
    )

    transformer = CrossModalTransformer(
        dim=128,
        depth=2,
        heads=4
    )

    ct_tokens, mri_tokens, H, W = transformer(
        ct_feat,
        mri_feat
    )

    print(
        "CT Tokens Shape:",
        ct_tokens.shape
    )

    print(
        "MRI Tokens Shape:",
        mri_tokens.shape
    )

    print(
        "Spatial Size:",
        H,
        W
    )

**SECTION-7 -> ADAPTIVE FUSION GATE**

In [ ]:
import torch
import torch.nn as nn

# =====================================
# SE Block (Channel Attention)
# =====================================

class SEBlock(nn.Module):

    def __init__(self, dim, reduction=8):

        super().__init__()

        self.fc = nn.Sequential(
            nn.Linear(dim, dim // reduction),
            nn.GELU(),
            nn.Linear(dim // reduction, dim),
            nn.Sigmoid()
        )

    def forward(self, x):

        # x : [B, N, C]

        w = x.mean(dim=1)      # [B, C]

        w = self.fc(w)         # [B, C]

        return x * w.unsqueeze(1)


# =====================================
# Adaptive Fusion Gate
# =====================================

class AdaptiveFusionGate(nn.Module):

    def __init__(self, dim=128):

        super().__init__()

        # Learnable modality weights
        self.alpha_ct = nn.Parameter(
            torch.ones(1)
        )

        self.alpha_mri = nn.Parameter(
            torch.ones(1)
        )

        # Fusion Projection
        self.fuse_proj = nn.Sequential(

            nn.Linear(
                dim * 2,
                dim
            ),

            nn.GELU(),

            nn.Dropout(0.1)

        )

        # SE Attention
        self.se = SEBlock(dim)

    def forward(
        self,
        ct_tokens,
        mri_tokens
    ):

        # Adaptive weighting

        ct = ct_tokens * self.alpha_ct

        mri = mri_tokens * self.alpha_mri

        # Concatenate

        fused = torch.cat(
            [ct, mri],
            dim=-1
        )

        # Project back to 128 dims

        fused = self.fuse_proj(
            fused
        )

        # Channel Attention

        fused = self.se(
            fused
        )

        return fused


# =====================================
# Test
# =====================================

if __name__ == "__main__":

    ct_tokens = torch.randn(
        1,
        1024,
        128
    )

    mri_tokens = torch.randn(
        1,
        1024,
        128
    )

    fusion = AdaptiveFusionGate(
        dim=128
    )

    fused_tokens = fusion(
        ct_tokens,
        mri_tokens
    )

    print(
        "Fused Tokens Shape:",
        fused_tokens.shape
    )

**SECTION-8 -> PROGRESSIVE DECODER**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# =====================================
# Channel Attention Module
# =====================================

class ChannelAttention(nn.Module):

    def __init__(self, channels):

        super().__init__()

        self.avg_pool = nn.AdaptiveAvgPool2d(1)

        self.attention = nn.Sequential(

            nn.Conv2d(
                channels,
                channels // 8,
                kernel_size=1
            ),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                channels // 8,
                channels,
                kernel_size=1
            ),

            nn.Sigmoid()

        )

    def forward(self, x):

        weights = self.attention(
            self.avg_pool(x)
        )

        return x * weights


# =====================================
# Progressive Decoder
# =====================================

class ProgressiveDecoder(nn.Module):

    def __init__(
        self,
        inchannels=128,
        outchannels=1
    ):

        super().__init__()

        # ---------------------------------
        # Stage 1 : 32x32 -> 64x64
        # ---------------------------------

        self.up1 = nn.Sequential(

            nn.ConvTranspose2d(
                inchannels,
                128,
                kernel_size=2,
                stride=2
            ),

            nn.BatchNorm2d(128),

            nn.LeakyReLU(
                0.1,
                inplace=True
            )

        )

        self.ca1 = ChannelAttention(128)

        # ---------------------------------
        # Skip Fusion
        # ---------------------------------

        self.skipfuse = nn.Sequential(

            nn.Conv2d(
                256,
                128,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(128),

            nn.LeakyReLU(
                0.1,
                inplace=True
            )

        )

        # ---------------------------------
        # Stage 2 : 64x64 -> 128x128
        # ---------------------------------

        self.up2 = nn.Sequential(

            nn.ConvTranspose2d(
                128,
                64,
                kernel_size=2,
                stride=2
            ),

            nn.BatchNorm2d(64),

            nn.LeakyReLU(
                0.1,
                inplace=True
            )

        )

        self.ca2 = ChannelAttention(64)

        # ---------------------------------
        # Stage 3 : 128x128 -> 256x256
        # ---------------------------------

        self.up3 = nn.Sequential(

            nn.ConvTranspose2d(
                64,
                32,
                kernel_size=2,
                stride=2
            ),

            nn.BatchNorm2d(32),

            nn.LeakyReLU(
                0.1,
                inplace=True
            )

        )

        self.ca3 = ChannelAttention(32)

        # ---------------------------------
        # Improved Reconstruction Block
        # ---------------------------------

        self.reconstruct = nn.Sequential(

            nn.Conv2d(
                32,
                32,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(32),

            nn.LeakyReLU(
                0.1,
                inplace=True
            ),

            nn.Conv2d(
                32,
                16,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(16),

            nn.LeakyReLU(
                0.1,
                inplace=True
            ),

            nn.Conv2d(
                16,
                outchannels,
                kernel_size=1
            ),
            nn.Sigmoid()

        )

    # =====================================
    # Forward Function
    # =====================================

    def forward(
        self,
        fusedtokens,
        encoderskip
    ):

        B, N, C = fusedtokens.shape

        # [B,1024,128] -> [B,128,32,32]

        x = fusedtokens.transpose(
            1,
            2
        ).contiguous().view(
            B,
            C,
            32,
            32
        )

        # Stage 1

        x = self.up1(x)

        x = self.ca1(x)

        # Match skip size automatically

        if encoderskip.shape[2:] != x.shape[2:]:

            encoderskip = F.interpolate(
                encoderskip,
                size=x.shape[2:],
                mode="bilinear",
                align_corners=False
            )

        # Skip Fusion

        x = torch.cat(
            [x, encoderskip],
            dim=1
        )

        x = self.skipfuse(x)

        # Stage 2

        x = self.up2(x)

        x = self.ca2(x)

        # Stage 3

        x = self.up3(x)

        x = self.ca3(x)

        # Reconstruction

        fusedimg = self.reconstruct(x)

        return fusedimg


# =====================================
# Create Decoder
# =====================================

decoder = ProgressiveDecoder()

print(
    "Progressive Decoder Loaded Successfully!"
)

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class AMTFNet(nn.Module):

    def __init__(self):
        super().__init__()

        # CT Encoder
        self.encoder_ct = DualBranchEncoder()

        # MRI Encoder
        self.encoder_mri = DualBranchEncoder()

        # Cross Modal Transformer
        self.transformer = CrossModalTransformer(
            dim=128,
            depth=2,
            heads=4
        )

        # Adaptive Fusion
        self.fusion = AdaptiveFusionGate(
            dim=128
        )

        # Progressive Decoder
        self.decoder = ProgressiveDecoder(
            inchannels=128,
            outchannels=1
        )

    def forward(self, ct_img, mri_img):

        # =====================================
        # Encoder Stage
        # =====================================

        ct_feat, ct_skip = self.encoder_ct(
            ct_img
        )

        mri_feat, mri_skip = self.encoder_mri(
            mri_img
        )

        # Shared Skip Feature
        encoder_skip = (
            ct_skip + mri_skip
        ) / 2

        # =====================================
        # Transformer Stage
        # =====================================

        ct_tokens, mri_tokens, H, W = (
            self.transformer(
                ct_feat,
                mri_feat
            )
        )

        # =====================================
        # Fusion Stage
        # =====================================

        fused_tokens = self.fusion(
            ct_tokens,
            mri_tokens
        )

        # =====================================
        # Decoder Stage
        # =====================================

        fused_img = self.decoder(
            fused_tokens,
            encoder_skip
        )

        return fused_img


# =====================================
# Initialize Model
# =====================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = AMTFNet().to(device)

print(
    "\n[SUCCESS] AMTFNet Model Compiled Successfully!\n"
)


# =====================================
# Test Pipeline
# =====================================

ct_batch, mri_batch = next(
    iter(trainloader)
)

ct_batch = ct_batch.to(device)

mri_batch = mri_batch.to(device)

with torch.no_grad():

    fused_output = model(
        ct_batch,
        mri_batch
    )

print(
    "\nPipeline Verification Output Shape:",
    fused_output.shape
)

**SECTION-9 -> LOSS FUNCTIONS**

In [ ]:
# =====================================
# Install SSIM Package
# =====================================

!pip install -q pytorch-msssim

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

from pytorch_msssim import ssim


# =====================================
# SSIM LOSS
# =====================================

def ssim_loss(fused, ct, mri):

    loss_ct = 1 - ssim(
        fused,
        ct,
        data_range=1.0,
        size_average=True
    )

    loss_mri = 1 - ssim(
        fused,
        mri,
        data_range=1.0,
        size_average=True
    )

    return loss_ct + loss_mri


# =====================================
# GRADIENT LOSS
# =====================================

def gradient_map(x):

    sobel_x = torch.tensor(
        [[-1,0,1],
         [-2,0,2],
         [-1,0,1]],
        dtype=torch.float32,
        device=x.device
    ).view(1,1,3,3)

    sobel_y = torch.tensor(
        [[-1,-2,-1],
         [0,0,0],
         [1,2,1]],
        dtype=torch.float32,
        device=x.device
    ).view(1,1,3,3)

    gx = F.conv2d(
        x,
        sobel_x,
        padding=1
    )

    gy = F.conv2d(
        x,
        sobel_y,
        padding=1
    )

    return torch.sqrt(
        gx**2 + gy**2 + 1e-6
    )


def gradient_loss(fused, ct, mri):

    grad_fused = gradient_map(fused)

    grad_target = torch.max(
        gradient_map(ct),
        gradient_map(mri)
    )

    return F.l1_loss(
        grad_fused,
        grad_target
    )


# =====================================
# EDGE LOSS
# =====================================

def laplacian_map(x):

    kernel = torch.tensor(
        [[0,-1,0],
         [-1,4,-1],
         [0,-1,0]],
        dtype=torch.float32,
        device=x.device
    ).view(1,1,3,3)

    return F.conv2d(
        x,
        kernel,
        padding=1
    )


def edge_loss(fused, ct, mri):

    fused_edge = laplacian_map(fused)

    target_edge = torch.max(
        laplacian_map(ct),
        laplacian_map(mri)
    )

    return F.l1_loss(
        fused_edge,
        target_edge
    )


# =====================================
# INTENSITY LOSS
# =====================================

def intensity_loss(fused, ct, mri):

    target = torch.max(
        ct,
        mri
    )

    return F.l1_loss(
        fused,
        target
    )


# =====================================
# PERCEPTUAL LOSS
# =====================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

vgg = models.vgg16(
    weights=models.VGG16_Weights.DEFAULT
)

vgg = vgg.features[:16]

vgg = vgg.to(device)

vgg.eval()

for param in vgg.parameters():
    param.requires_grad = False


class PerceptualLoss(nn.Module):

    def __init__(self):

        super().__init__()

        self.vgg = vgg

    def forward(
        self,
        fused,
        ct,
        mri
    ):

        fused3 = fused.repeat(
            1,3,1,1
        )

        ct3 = ct.repeat(
            1,3,1,1
        )

        mri3 = mri.repeat(
            1,3,1,1
        )

        feat_f = self.vgg(fused3)

        feat_ct = self.vgg(ct3)

        feat_mri = self.vgg(mri3)

        target = (
            feat_ct + feat_mri
        ) / 2

        return F.mse_loss(
            feat_f,
            target
        )


perceptual = PerceptualLoss()


# =====================================
# INFORMATION LOSS
# =====================================

def information_loss(fused):

    p = torch.softmax(
        fused.view(
            fused.size(0),
            -1
        ),
        dim=1
    )

    entropy = -torch.sum(
        p * torch.log(
            p + 1e-8
        ),
        dim=1
    )

    return torch.mean(
        entropy
    )


# =====================================
# TOTAL LOSS
# =====================================

def total_loss(fused, ct, mri):

    l_ssim = ssim_loss(fused, ct, mri)

    l_grad = gradient_loss(fused, ct, mri)

    l_edge = edge_loss(fused, ct, mri)

    l_intensity = intensity_loss(fused, ct, mri)

    l_perc = perceptual(fused, ct, mri)

    l_info = information_loss(fused)

    loss = (
        2.0 * l_ssim +
        1.0 * l_grad +
        1.5 * l_edge +
        0.5 * l_intensity +
        0.3 * l_perc -
        0.01 * l_info
    )

    return loss


# =====================================
# OPTIMIZER
# =====================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

print(
    "Enhanced Loss System Loaded Successfully"
)

**SECTION-10 -> TRAINING LOOP**

In [ ]:
# =====================================
# Section 10: Enhanced Training Loop
# =====================================

import torch

epochs = 200

best_loss = float("inf")

# Learning Rate Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=10
)
for epoch in range(epochs):

    model.train()

    running_loss = 0.0

    for ct_batch, mri_batch in trainloader:

        ct_batch = ct_batch.to(device)
        mri_batch = mri_batch.to(device)

        optimizer.zero_grad()

        # -------------------------
        # Forward Pass
        # -------------------------

        fused_output = model(
            ct_batch,
            mri_batch
        )

        # -------------------------
        # Loss Computation
        # -------------------------

        loss = total_loss(
            fused_output,
            ct_batch,
            mri_batch
        )

        # -------------------------
        # Backpropagation
        # -------------------------

        loss.backward()

        # Gradient Clipping
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        running_loss += loss.item()

    # -------------------------
    # Epoch Statistics
    # -------------------------

    epoch_loss = (
        running_loss /
        len(trainloader)
    )

    # Scheduler Update
    scheduler.step(epoch_loss)

    # Save Best Model
    if epoch_loss < best_loss:

        best_loss = epoch_loss

        torch.save(
            model.state_dict(),
            "best_AMTFNet.pth"
        )

        save_status = " [BEST SAVED]"

    else:

        save_status = ""

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {epoch_loss:.6f}"
        f"{save_status}"
    )

print("\nTraining Completed Successfully!")

print(
    f"Best Training Loss: "
    f"{best_loss:.6f}"
)

In [ ]:
# Load Best Saved Model

model.load_state_dict(
    torch.load(
        "best_AMTFNet.pth",
        map_location=device
    )
)

model.eval()

print("Model loaded successfully!")

In [ ]:
print(
    "Alpha CT item : ",model.fusion.alpha_ct.item(),
    "Alpha MRI item : ",model.fusion.alpha_mri.item()
)

**SECTION-11 -> TESTING**

In [ ]:
# =====================================
# Testing & Save Fused Images
# =====================================

import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------------
# Create Output Folder
# -------------------------------------

output_folder = "Fused_Output"

os.makedirs(
    output_folder,
    exist_ok=True
)

model.eval()

display_count = 0

# -------------------------------------
# Generate Fused Images
# -------------------------------------

with torch.no_grad():

    for i, (ct_batch, mri_batch) in enumerate(trainloader):

        ct_batch = ct_batch.to(device)
        mri_batch = mri_batch.to(device)

        # ---------------------------------
        # Fusion
        # ---------------------------------

        fused = model(
            ct_batch,
            mri_batch
        )

        # ---------------------------------
        # Convert to NumPy
        # ---------------------------------

        fused_img = fused[0, 0].cpu().numpy()

        fused_img = np.clip(
            fused_img,
            0,
            1
        )

        # ---------------------------------
        # Convert to uint8
        # ---------------------------------

        fused_uint8 = (
            fused_img * 255
        ).astype(np.uint8)

        # ---------------------------------
        # Noise Removal
        # ---------------------------------

        fused_denoised = cv2.fastNlMeansDenoising(
            fused_uint8,
            None,
            h=10,
            templateWindowSize=7,
            searchWindowSize=21
        )

        # ---------------------------------
        # Gamma Correction
        # ---------------------------------

        gamma = 0.9

        fused_enhanced = np.power(
            fused_denoised.astype(np.float32) / 255.0,
            gamma
        )

        fused_enhanced = (
            fused_enhanced * 255
        ).astype(np.uint8)

        # ---------------------------------
        # Save Image
        # ---------------------------------

        save_path = os.path.join(
            output_folder,
            f"Fused_{i+1}.png"
        )

        cv2.imwrite(
            save_path,
            fused_enhanced
        )

        # ---------------------------------
        # Display First 5 Images
        # ---------------------------------

        if display_count < 5:

            plt.figure(figsize=(15,5))

            plt.subplot(1,3,1)
            plt.imshow(
                ct_batch[0,0].cpu(),
                cmap="gray"
            )
            plt.title("CT Image")
            plt.axis("off")

            plt.subplot(1,3,2)
            plt.imshow(
                mri_batch[0,0].cpu(),
                cmap="gray"
            )
            plt.title("MRI Image")
            plt.axis("off")

            plt.subplot(1,3,3)
            plt.imshow(
                fused_enhanced,
                cmap="gray"
            )
            plt.title("Fused Image")
            plt.axis("off")

            plt.tight_layout()
            plt.show()

            display_count += 1

print(
    f"\nSuccessfully generated and saved "
    f"{len(trainloader)} fused images."
)

print(
    f"Images stored in: {output_folder}"
)

In [ ]:
with torch.no_grad():

    fused_output = model(
        ct_batch,
        mri_batch
    )

print("Min :", fused_output.min().item())
print("Max :", fused_output.max().item())
print("Mean:", fused_output.mean().item())

In [ ]:
print("Alpha CT item : ",model.fusion.alpha_ct.item())
print("Alpha MRI item : ",model.fusion.alpha_mri.item())

**SECTION-12 -> XAI (GRAND - CAM)**

In [ ]:
class GradCAM:

    def __init__(self, model, target_layer):

        self.model = model
        self.target_layer = target_layer

        self.activations = None
        self.gradients = None

        self.forward_hook = target_layer.register_forward_hook(
            self.save_activation
        )

        self.backward_hook = target_layer.register_full_backward_hook(
            self.save_gradient
        )

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate(self):

        grads = self.gradients
        acts = self.activations

        weights = grads.mean(dim=(2,3), keepdim=True)

        cam = (weights * acts).sum(dim=1)

        cam = F.relu(cam)

        cam = cam.squeeze().detach().cpu().numpy()

        cam = (
            cam - cam.min()
        ) / (
            cam.max() - cam.min() + 1e-8
        )

        return cam

In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

# Folder to save Grad-CAM results
save_folder = "GradCAM_Output"
os.makedirs(save_folder, exist_ok=True)

# Initialize GradCAM
gradcam = GradCAM(
    model,
    model.encoder_ct.conv1[0]
)

model.eval()

display_count = 0

for idx, (ct_batch, mri_batch) in enumerate(trainloader):

    batch_size = ct_batch.size(0)

    for b in range(batch_size):

        ct_sample = ct_batch[b:b+1].to(device)
        mri_sample = mri_batch[b:b+1].to(device)

        # Forward pass
        output = model(ct_sample, mri_sample)

        score = output.mean()

        model.zero_grad()
        score.backward()

        # Generate CAM
        cam = gradcam.generate()

        cam = cv2.resize(cam, (256, 256))

        ct_img = ct_sample.squeeze().cpu().numpy()

        # Heatmap
        heatmap = np.uint8(255 * cam)

        heatmap_color = cv2.applyColorMap(
            heatmap,
            cv2.COLORMAP_JET
        )

        # Normalize CT image
        ct_img_uint8 = (
            (ct_img - ct_img.min()) /
            (ct_img.max() - ct_img.min() + 1e-8) * 255
        ).astype(np.uint8)

        ct_rgb = cv2.cvtColor(
            ct_img_uint8,
            cv2.COLOR_GRAY2RGB
        )

        # Overlay
        overlay = cv2.addWeighted(
            ct_rgb,
            0.6,
            heatmap_color,
            0.4,
            0
        )

        # Save image
        save_path = os.path.join(
            save_folder,
            f"gradcam_{idx}_{b}.png"
        )

        cv2.imwrite(
            save_path,
            cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
        )

        # Display only first 5 images
        if display_count < 5:

            plt.figure(figsize=(18,6))

            plt.subplot(1,3,1)
            plt.imshow(ct_img, cmap='gray')
            plt.title("Original CT")
            plt.axis("off")

            plt.subplot(1,3,2)
            plt.imshow(cam, cmap='jet')
            plt.title("Grad-CAM")
            plt.axis("off")

            plt.subplot(1,3,3)
            plt.imshow(overlay)
            plt.title("Grad-CAM Overlay")
            plt.axis("off")

            plt.show()

            display_count += 1

print(f"All Grad-CAM images saved in: {save_folder}")

**SECTION-13 -> EVALUATION METRICS**

In [ ]:
# ==========================================
# FUSION EVALUATION
# ==========================================

import torch
import cv2
import numpy as np
from skimage.metrics import structural_similarity as ssim
from sklearn.metrics import mutual_info_score
from math import log10

model.eval()

# ==========================================
# Metric Storage
# ==========================================

EN_list = []
SD_list = []
SF_list = []
AG_list = []

MI_CT_list = []
MI_MRI_list = []

SSIM_CT_list = []
SSIM_MRI_list = []

PSNR_CT_list = []
PSNR_MRI_list = []

CC_CT_list = []
CC_MRI_list = []

# ==========================================
# PSNR Function
# ==========================================

def psnr(img1, img2):

    mse = np.mean(
        (img1.astype(np.float64) -
         img2.astype(np.float64)) ** 2
    )

    if mse == 0:
        return 100

    return 20 * log10(
        255.0 / np.sqrt(mse)
    )

# ==========================================
# Evaluation
# ==========================================

with torch.no_grad():

    for ct_batch, mri_batch in trainloader:

        ct_batch = ct_batch.to(device)
        mri_batch = mri_batch.to(device)

        fused_batch = model(
            ct_batch,
            mri_batch
        )

        batch_size = ct_batch.size(0)

        for i in range(batch_size):

            # --------------------------
            # Convert to NumPy
            # --------------------------

            ct = ct_batch[i,0].cpu().numpy()
            mri = mri_batch[i,0].cpu().numpy()
            fused = fused_batch[i,0].cpu().numpy()

            ct = (
                (ct - ct.min())
                /
                (ct.max() - ct.min() + 1e-8)
                * 255
            ).astype(np.uint8)

            mri = (
                (mri - mri.min())
                /
                (mri.max() - mri.min() + 1e-8)
                * 255
            ).astype(np.uint8)

            fused = np.clip(
                fused,
                0,
                1
            )

            fused = (
                fused * 255
            ).astype(np.uint8)

            # ==================================
            # Entropy
            # ==================================

            hist = cv2.calcHist(
                [fused],
                [0],
                None,
                [256],
                [0,256]
            )

            hist = hist / hist.sum()
            hist = hist[hist > 0]

            EN = -np.sum(
                hist * np.log2(hist)
            )

            EN_list.append(EN)

            # ==================================
            # Standard Deviation
            # ==================================

            SD = np.std(fused)
            SD_list.append(SD)

            # ==================================
            # Spatial Frequency
            # ==================================

            rf = np.sqrt(
                np.mean(
                    np.diff(
                        fused.astype(np.float64),
                        axis=0
                    )**2
                )
            )

            cf = np.sqrt(
                np.mean(
                    np.diff(
                        fused.astype(np.float64),
                        axis=1
                    )**2
                )
            )

            SF = np.sqrt(
                rf**2 + cf**2
            )

            SF_list.append(SF)

            # ==================================
            # Average Gradient
            # ==================================

            gx = np.diff(
                fused.astype(np.float64),
                axis=1
            )

            gy = np.diff(
                fused.astype(np.float64),
                axis=0
            )

            gx = gx[:-1,:]
            gy = gy[:,:-1]

            AG = np.mean(
                np.sqrt(
                    (gx**2 + gy**2)/2
                )
            )

            AG_list.append(AG)

            # ==================================
            # Mutual Information
            # ==================================

            MI_CT = mutual_info_score(
                ct.ravel(),
                fused.ravel()
            )

            MI_MRI = mutual_info_score(
                mri.ravel(),
                fused.ravel()
            )

            MI_CT_list.append(MI_CT)
            MI_MRI_list.append(MI_MRI)

            # ==================================
            # SSIM
            # ==================================

            SSIM_CT = ssim(
                ct,
                fused,
                data_range=255
            )

            SSIM_MRI = ssim(
                mri,
                fused,
                data_range=255
            )

            SSIM_CT_list.append(
                SSIM_CT
            )

            SSIM_MRI_list.append(
                SSIM_MRI
            )

            # ==================================
            # PSNR
            # ==================================

            PSNR_CT = psnr(
                ct,
                fused
            )

            PSNR_MRI = psnr(
                mri,
                fused
            )

            PSNR_CT_list.append(
                PSNR_CT
            )

            PSNR_MRI_list.append(
                PSNR_MRI
            )

            # ==================================
            # Correlation Coefficient
            # ==================================

            CC_CT = np.corrcoef(
                ct.flatten(),
                fused.flatten()
            )[0,1]

            CC_MRI = np.corrcoef(
                mri.flatten(),
                fused.flatten()
            )[0,1]

            CC_CT_list.append(CC_CT)
            CC_MRI_list.append(CC_MRI)

# ==========================================
# Final Averages
# ==========================================

avg_en = np.mean(EN_list)
avg_sd = np.mean(SD_list)
avg_sf = np.mean(SF_list)
avg_ag = np.mean(AG_list)

avg_mi_ct = np.mean(MI_CT_list)
avg_mi_mri = np.mean(MI_MRI_list)

avg_ssim_ct = np.mean(SSIM_CT_list)
avg_ssim_mri = np.mean(SSIM_MRI_list)

avg_psnr_ct = np.mean(PSNR_CT_list)
avg_psnr_mri = np.mean(PSNR_MRI_list)

avg_cc_ct = np.mean(CC_CT_list)
avg_cc_mri = np.mean(CC_MRI_list)

# ==========================================
# OVERALL FUSION ACCURACY
# ==========================================

fusion_accuracy = (
    (avg_ssim_ct + avg_ssim_mri)
    / 2
) * 100

# ==========================================
# Results
# ==========================================

print("\n========== FUSION METRICS ==========\n")

print(f"Entropy (EN)              : {avg_en:.4f}")
print(f"Standard Deviation (SD)   : {avg_sd:.4f}")
print(f"Spatial Frequency (SF)    : {avg_sf:.4f}")
print(f"Average Gradient (AG)     : {avg_ag:.4f}")

print("\n----- Mutual Information -----")
print(f"MI (CT-Fused)             : {avg_mi_ct:.4f}")
print(f"MI (MRI-Fused)            : {avg_mi_mri:.4f}")
print(f"Total MI                  : {(avg_mi_ct + avg_mi_mri):.4f}")

print("\n----- SSIM -----")
print(f"SSIM (CT-Fused)           : {avg_ssim_ct:.4f}")
print(f"SSIM (MRI-Fused)          : {avg_ssim_mri:.4f}")
print(f"Average SSIM              : {((avg_ssim_ct + avg_ssim_mri)/2):.4f}")


print("\n----- PSNR -----")
print(f"PSNR (CT-Fused)           : {avg_psnr_ct:.4f} dB")
print(f"PSNR (MRI-Fused)          : {avg_psnr_mri:.4f} dB")
print(f"Average PSNR              : {((avg_psnr_ct + avg_psnr_mri)/2):.4f} dB")

print("\n----- Correlation Coefficient -----")
print(f"CC (CT-Fused)             : {avg_cc_ct:.4f}")
print(f"CC (MRI-Fused)            : {avg_cc_mri:.4f}")
print(f"Average CC                : {((avg_cc_ct + avg_cc_mri)/2):.4f}")

print("\n----- OVERALL FUSION ACCURACY -----")
print(f"Fusion Accuracy           : {fusion_accuracy:.2f}%")

print("\n===================================")

In [ ]:
# ==========================================
# Fusion Performance Score
# ==========================================

FPS = (
    (
        ((avg_ssim_ct + avg_ssim_mri) / 2)
        +
        ((avg_cc_ct + avg_cc_mri) / 2)
        +
        ((avg_mi_ct + avg_mi_mri) / 3)
    ) / 3
) * 100
print("\n----- Fusion Performance Score -----")
print(f"Fusion Performance Score : {FPS:.2f}%")